# 2. Router 평가

학습에 사용하지 않은 project-disjoint test split에서 Anchor/Rare 구조를 평가합니다. Utility outcome test가 있으면 실제 Expert×Model 성공 coverage, 비용, regret도 함께 계산합니다.

In [1]:
import json
import sys
from pathlib import Path
from pprint import pprint

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
DATA_DIR = ROOT / 'data' / 'phase2e'
ARTIFACT_DIR = ROOT / 'artifacts' / 'phase2e'
RESULTS_DIR = ROOT / 'results' / 'router'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

from llm_security.datasets import load_router_samples_jsonl, load_utility_samples_jsonl
from llm_security.models import ExpertFamily, to_dict
from llm_security.routing import AnchorRareRouter, BudgetedUtilityRouter

In [2]:
anchor_path = ARTIFACT_DIR / 'router_anchor_rare_v1.pkl'
test_path = DATA_DIR / 'semantic' / 'router_test.jsonl'
if not anchor_path.exists():
    raise FileNotFoundError('먼저 01_train_router.ipynb를 실행하세요.')
router = AnchorRareRouter.load(anchor_path)
test_samples = load_router_samples_jsonl(test_path)
anchor_metrics = router.evaluate(test_samples)

fixed_anchors = {ExpertFamily.MEMORY_BOUNDS, ExpertFamily.CONTROL_STATE_ERROR}
fixed_exact_coverage = sum(
    set(sample.labels).issubset(fixed_anchors) for sample in test_samples
) / len(test_samples)
summary = {
    'artifact': str(anchor_path),
    'fixed_top2_exact_coverage': fixed_exact_coverage,
    'anchor_rare': to_dict(anchor_metrics),
}
pprint(summary)

{'anchor_rare': {'average_experts_per_candidate': 2.8890814558058926,
                 'exact_coverage': 0.9965337954939342,
                 'expert_coverage': 0.9965337954939342,
                 'llm_calls_saved_vs_all_six': 1795,
                 'rare_precision': 0.1267056530214425,
                 'rare_recall': 0.9701492537313433,
                 'rare_trigger_rate': 0.8890814558058926,
                 'sample_count': 577},
 'artifact': 'C:\\Users\\junhyun111\\Desktop\\llm-security\\artifacts\\phase2e\\router_anchor_rare_v1.pkl',
 'fixed_top2_exact_coverage': 0.8838821490467937}


In [3]:
utility_path = ARTIFACT_DIR / 'router_utility_v1.pkl'
utility_test_path = ROOT / 'data' / 'utility' / 'outcomes_test.jsonl'
if utility_path.exists() and utility_test_path.exists():
    utility_router = BudgetedUtilityRouter.load(utility_path)
    utility_test = load_utility_samples_jsonl(utility_test_path)
    utility_metrics = utility_router.evaluate(utility_test)
    summary['utility'] = to_dict(utility_metrics)
else:
    summary['utility'] = {
        'evaluated': False,
        'reason': 'Utility artifact or outcomes_test.jsonl not found',
    }
pprint(summary['utility'])

{'evaluated': False,
 'reason': 'Utility artifact or outcomes_test.jsonl not found'}


## 해석 기준

rare_recall과 success_coverage를 우선 확인합니다. average_experts 또는 average_assignments와 비용은 같은 coverage를 달성하는 설정끼리 비교해야 합니다. family Top-1 accuracy는 새 Router의 최적화 목표가 아닙니다.

In [4]:
metrics_path = RESULTS_DIR / 'router_metrics.json'
metrics_path.write_text(
    json.dumps(summary, ensure_ascii=False, indent=2) + '\n', encoding='utf-8'
)
print('saved:', metrics_path)

saved: C:\Users\junhyun111\Desktop\llm-security\results\router\router_metrics.json
